# SpillTrace — U-Net oil-spill segmentation (Phase 7, optional)

Trains a binary oil/no-oil segmentation model on the SAR-SOS dataset in `ml/data/`
(PALSAR + Sentinel-1 patches, 256x256, `images/{train,val}` + `masks/{train,val}`).

This is an upgrade layered on top of the classical detector in
`backend/app/detection/classical.py`, never a replacement for it — see `plan.md`
Phase 7. Export weights to `ml/weights/unet_best.pth` and load them behind the
`method: "unet"` flag in `detection/unet.py`. Only ship this if it beats the
classical baseline's IoU (0.878) on the held-out split.

Runs locally on a 4GB-VRAM GPU (RTX 2050) at this patch size, or on Colab if you
want a bigger GPU / faster iteration.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0), "-", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Config

Batch size 8 and no gradient accumulation is sized for 4GB VRAM at 256x256 with a
ResNet34 encoder + AMP. If you hit `CUDA out of memory`, drop `BATCH_SIZE` to 4
before touching anything else.

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-4
ENCODER = "resnet34"
ENCODER_WEIGHTS = "imagenet"
PATIENCE = 6  # early stop if val IoU doesn't improve for this many epochs

## Dataset

Masks are pure binary PNGs (black = background, white = oil) — see the pixel-value
check done in-session before this notebook was written. Threshold at 127 to make
sure lossy/anti-aliased edges don't leak fractional mask values into training.

In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, images_dir: Path, masks_dir: Path, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.filenames = sorted(p.name for p in images_dir.iterdir() if p.suffix == ".png")
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        image = cv2.imread(str(self.images_dir / name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.masks_dir / name), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        return image, mask.unsqueeze(0).float()


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_ds = OilSpillDataset(DATA_DIR / "images/train", DATA_DIR / "masks/train", train_transform)
val_ds = OilSpillDataset(DATA_DIR / "images/val", DATA_DIR / "masks/val", val_transform)

print("train:", len(train_ds), "val:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## Sanity check — visualize a batch before training anything

In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

imgs, masks = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(denormalize(imgs[i]).permute(1, 2, 0).numpy())
    axes[0, i].set_title("image")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks[i, 0].numpy(), cmap="gray")
    axes[1, i].set_title("mask")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## Model, loss, optimizer

Oil pixels are a small minority in most patches, so plain BCE biases toward
predicting all-background. BCE + Dice combined fixes that without needing a
hand-tuned class weight.

In [ ]:
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=1,
).to(DEVICE)

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)

def criterion(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.1f}M parameters")

## Metrics

In [ ]:
@torch.no_grad()
def iou_score(logits, targets, threshold=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = (preds + targets).clamp(0, 1).sum(dim=(1, 2, 3))
    return ((intersection + eps) / (union + eps)).mean().item()

## Training loop

Mixed precision (AMP) roughly halves VRAM use and speeds up training on the
RTX 2050. Checkpoints the best val-IoU epoch to `weights/unet_best.pth` and
early-stops after `PATIENCE` epochs with no improvement.

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_iou": []}
best_iou = 0.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS} [train]", leave=False):
        imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_ds)

    model.eval()
    val_running_loss = 0.0
    val_running_iou = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"epoch {epoch}/{EPOCHS} [val]", leave=False):
            imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, masks)
            val_running_loss += loss.item() * imgs.size(0)
            val_running_iou += iou_score(logits, masks) * imgs.size(0)

    val_loss = val_running_loss / len(val_ds)
    val_iou = val_running_iou / len(val_ds)
    scheduler.step(val_iou)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_iou={val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "encoder": ENCODER,
            "encoder_weights": ENCODER_WEIGHTS,
            "img_size": IMG_SIZE,
            "val_iou": val_iou,
            "epoch": epoch,
        }, WEIGHTS_DIR / "unet_best.pth")
        print(f"  -> saved new best (val_iou={val_iou:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"early stopping — no improvement for {PATIENCE} epochs")
            break

print(f"\nbest val IoU: {best_iou:.4f}  (classical baseline: 0.878)")

## Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("loss")
axes[0].legend()
axes[1].plot(history["val_iou"], label="val IoU")
axes[1].axhline(0.878, color="gray", linestyle="--", label="classical baseline")
axes[1].set_title("val IoU")
axes[1].legend()
plt.tight_layout()
plt.show()

## Qualitative check — predictions on a few val patches

In [ ]:
checkpoint = torch.load(WEIGHTS_DIR / "unet_best.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

imgs, masks = next(iter(val_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(imgs.to(DEVICE))).cpu()

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i in range(4):
    axes[0, i].imshow(denormalize(imgs[i]).permute(1, 2, 0).numpy())
    axes[0, i].set_title("image")
    axes[1, i].imshow(masks[i, 0].numpy(), cmap="gray")
    axes[1, i].set_title("ground truth")
    axes[2, i].imshow(preds[i, 0].numpy() > 0.5, cmap="gray")
    axes[2, i].set_title("prediction")
    for r in range(3):
        axes[r, i].axis("off")
plt.tight_layout()
plt.show()

print(f"Saved weights: {WEIGHTS_DIR / 'unet_best.pth'}")
print("Next: wire this checkpoint into backend/app/detection/unet.py behind the")
print("'method: unet' flag, and only make it the default if its held-out mIoU")
print("beats the classical detector (0.878) — per plan.md Phase 7.")